In [1]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Demonstrate extracting pixels from an image as features with
# ee.Image.sample(), and show how the features are aligned with the pixels.

# An image with one band of elevation data.
image = ee.Image('CGIAR/SRTM90_V4')
vis_min = 1620
vis_max = 1650
m = geemap.Map()
m.add_layer(image, {'min': vis_min, 'max': vis_max}, 'SRTM')

# Region to sample.
region = ee.Geometry.Polygon(
    [[
        [-110.006, 40.002],
        [-110.006, 39.999],
        [-109.995, 39.999],
        [-109.995, 40.002],
    ]],
    None,
    False,
)
# Show region on the map.
m.set_center(-110, 40, 16)

m.add_layer(ee.FeatureCollection([region]).style(color='00FF0022'))

# Perform sampling convert image pixels to features.
samples = image.sample(
    region=region,
    # Default (False) is no geometries in the output.
    # When set to True, each feature has a Point geometry at the center of the
    # image pixel.
    geometries=True,
    # The scale is not specified, so the resolution of the image will be used,
    # and there is a feature for every pixel. If we give a scale parameter, the
    # image will be resampled and there will be more or fewer features.
    #
    # scale=200,
)


def scale_point_size(feature):
  elevation = feature.getNumber('elevation')
  point_size = elevation.unitScale(vis_min, vis_max).multiply(15)
  feature.set('style', {'pointSize': point_size})
  return feature


# Visualize sample data using ee.FeatureCollection.style().
styled = samples.map(scale_point_size).style(
    color='000000FF',
    fillColor='00000000',
    styleProperty='style',
    neighborhood=6,  # increase to correctly draw large points
)
m.add_layer(styled)
display(m)

# Each sample feature has a point geometry and a property named 'elevation'
# corresponding to the band named 'elevation' of the image. If there are
# multiple bands they will become multiple properties. This will print:
#
# geometry: Point (-110.01, 40.00)
# properties:
#   elevation: 1639
display(samples.first())

Map(center=[40, -110], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…